# 🧠 Pelatihan Model PhishShield

Notebook ini melatih model ML kecil yang menilai **peluang phishing sebuah URL** dari ciri-ciri URL-nya (rancangan bagian 6), lalu mengekspornya ke `extension/model/model.json` supaya bisa dijalankan **langsung di browser**.

**Sebelum mulai:**
1. Install library: `pip install -r requirements.txt`
2. Siapkan dataset: `python prepare_data.py --source ...` → menghasilkan `data/processed/urls.csv` (lihat `ml/README.md`)
3. Jalankan semua sel dari atas ke bawah.

**Aturan emas:** hanya pakai fitur yang bisa dihitung ulang di ekstensi. Semua fitur di sini dihitung oleh `features.py`, kembaran `extension/features.js`.

In [ ]:
import sys
import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

# Notebook bisa dibuka dari folder ml/ atau dari root repo
ML_DIR = Path.cwd() if (Path.cwd() / 'features.py').exists() else Path.cwd() / 'ml'
sys.path.insert(0, str(ML_DIR))

import features as F
import export_model as EX

DATA_PATH = ML_DIR / 'data' / 'processed' / 'urls.csv'
ARTIFACTS = ML_DIR / 'artifacts'
ARTIFACTS.mkdir(exist_ok=True)
RANDOM_STATE = 42

print('Folder ML :', ML_DIR)
print('Dataset   :', DATA_PATH)

## 1. Muat dataset

Format: `url, label, source` dengan `label` 1 = phishing, 0 = legit (dibuat oleh `prepare_data.py`).

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(f'{DATA_PATH} belum ada. Jalankan prepare_data.py dulu (lihat ml/README.md).')

df = pd.read_csv(DATA_PATH, dtype={'url': str})
print(f'Jumlah URL: {len(df):,}')
print(df['label'].map({0: 'legit (0)', 1: 'phishing (1)'}).value_counts())
df.sample(10, random_state=RANDOM_STATE)

## 2. Pilih fitur & ekstrak

URL diubah jadi angka oleh `features.extract_features()`. Fungsi ini meniru normalisasi URL di browser (huruf kecil, punycode, port default dibuang, dll.), jadi angka yang dipakai saat training sama dengan angka yang dihitung ekstensi dari `location.href`.

In [ ]:
# Fitur inti dari rancangan (bagian 6.2). Semuanya bisa dihitung ulang di extension/features.js.
FEATURES = list(F.CORE_FEATURES)

# Fitur tambahan (opsional): aktifkan lalu bandingkan hasil evaluasinya.
# FEATURES += ['HostLength', 'PathLength', 'NumDigits', 'NumQueryParams', 'HasPunycode']

# Kalau dataset-mu banyak URL tanpa http:// atau https://, IsHTTPS tidak bisa dipercaya:
# FEATURES.remove('IsHTTPS')

print('Fitur dipakai:', FEATURES)

In [ ]:
rows = F.extract_many(df['url'])
valid = np.array([r is not None for r in rows])
print(f'URL gagal diparse (dibuang): {(~valid).sum():,}')

feat_df = pd.DataFrame([r for r in rows if r is not None], index=df.index[valid])
data = pd.concat([df.loc[valid, ['url', 'label']], feat_df], axis=1)
X = data[FEATURES].astype(np.int64)
y = data['label'].astype(int)

# Rata-rata tiap fitur per kelas: fitur yang bagus nilainya beda jauh antara legit & phishing
data.groupby('label')[F.FEATURE_NAMES].mean().T.rename(columns={0: 'rata-rata legit', 1: 'rata-rata phishing'})

## 3. Bagi data latih & uji

In [ ]:
X_train, X_test, y_train, y_test, url_train, url_test = train_test_split(
    X, y, data['url'], test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
print(f'Data latih: {len(X_train):,}  |  Data uji: {len(X_test):,}')

## 4. Latih model

Dua kandidat sesuai rancangan:
- **Decision Tree**: satu pohon, paling mudah dijelaskan, file kecil.
- **Random Forest**: banyak pohon, biasanya lebih akurat, file lebih besar.

`max_depth`, `n_estimators`, dan `min_samples_leaf` menentukan ukuran `model.json`. Usahakan di bawah ±5 MB supaya ringan di browser.

In [ ]:
models = {
    'Decision Tree': DecisionTreeClassifier(
        max_depth=8, min_samples_leaf=20, class_weight='balanced', random_state=RANDOM_STATE
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=60, max_depth=12, min_samples_leaf=10,
        class_weight='balanced', n_jobs=-1, random_state=RANDOM_STATE
    ),
}

for name, m in models.items():
    m.fit(X_train, y_train)
    n_nodes = sum(e.tree_.node_count for e in EX.estimators_of(m))
    print(f'✓ {name} selesai dilatih ({n_nodes:,} node)')

## 5. Evaluasi

Jangan hanya melihat akurasi (rancangan 6.4):
- **Precision**: dari semua yang dianggap phishing, berapa yang benar phishing.
- **Recall**: dari semua phishing, berapa yang tertangkap.
- **False positive rate**: situs aman yang salah dianggap berbahaya. Bikin pengguna kesal, harus serendah mungkin.
- **False negative rate**: phishing yang lolos. Ini bahaya sebenarnya.

In [ ]:
def positive_proba(model, X):
    return model.predict_proba(X)[:, EX.positive_index(model)]


def evaluate(model, X, y, threshold=0.5):
    proba = positive_proba(model, X)
    pred = (proba >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return {
        'accuracy': accuracy_score(y, pred),
        'precision': precision_score(y, pred, zero_division=0),
        'recall': recall_score(y, pred, zero_division=0),
        'f1': f1_score(y, pred, zero_division=0),
        'roc_auc': roc_auc_score(y, proba),
        'false_positive_rate': fp / (fp + tn) if fp + tn else 0.0,
        'false_negative_rate': fn / (fn + tp) if fn + tp else 0.0,
    }


results = pd.DataFrame({name: evaluate(m, X_test, y_test) for name, m in models.items()}).T
results.style.format('{:.4f}')

In [ ]:
fig, axes = plt.subplots(1, len(models), figsize=(5 * len(models), 4))
for ax, (name, m) in zip(np.atleast_1d(axes), models.items()):
    ConfusionMatrixDisplay.from_estimator(
        m, X_test, y_test, display_labels=['legit', 'phishing'], ax=ax, colorbar=False
    )
    ax.set_title(name)
plt.tight_layout()
plt.show()

## 6. Pilih model

Pilih yang false positive rate-nya rendah dengan recall yang masih bagus.

In [ ]:
CHOSEN = 'Random Forest'  # ganti ke 'Decision Tree' kalau ingin model paling sederhana
model = models[CHOSEN]
print(f'Model terpilih: {CHOSEN}')
print()
print(classification_report(y_test, model.predict(X_test), target_names=['legit', 'phishing'], digits=4))

## 7. Analisis ambang (threshold)

Di ekstensi, skor risiko = peluang × 100 (dibulatkan), lalu:

| Skor | Level |
|---|---|
| < `warn` | 🟢 Aman, tanpa gangguan |
| ≥ `warn` | 🟡 Waspada: banner kecil, form mulai dipantau |
| ≥ `danger` | 🔴 Berbahaya |

Titik awal rancangan: `warn = 40`, `danger = 70`. Tabel di bawah menunjukkan dampak tiap ambang pada data uji.

Catatan: ekstensi juga menambah poin heuristik (redirect, link pendek, umur domain) di atas skor model, lihat `extension/config.js`.

In [ ]:
proba_test = positive_proba(model, X_test)
y_true = y_test.to_numpy()
score_test = np.floor(proba_test * 100 + 0.5)  # sama dengan Math.round di JavaScript

rows = []
for t in range(30, 95, 5):
    pred = (score_test >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    rows.append({
        'ambang (%)': t,
        'false positive rate': fp / (fp + tn) if fp + tn else 0.0,
        'recall (phishing tertangkap)': tp / (tp + fn) if tp + fn else 0.0,
        'precision': tp / (tp + fp) if tp + fp else np.nan,
    })
pd.DataFrame(rows).set_index('ambang (%)').style.format('{:.4f}')

In [ ]:
# Saran ambang berdasarkan batas false positive yang masih bisa diterima
TARGET_FPR_WARN = 0.05    # maksimal 5% situs legit mendapat banner Waspada
TARGET_FPR_DANGER = 0.01  # maksimal 1% situs legit mendapat label Berbahaya

legit_scores = score_test[y_true == 0]
suggest_warn = int(np.clip(np.quantile(legit_scores, 1 - TARGET_FPR_WARN) + 1, 1, 99))
suggest_danger = int(np.clip(max(np.quantile(legit_scores, 1 - TARGET_FPR_DANGER) + 1, suggest_warn + 1), 2, 100))
print(f'Saran ambang: warn = {suggest_warn}, danger = {suggest_danger}')

# Default = titik awal rancangan. Ganti dengan saran di atas kalau setuju.
THRESHOLDS = {'warn': 40, 'danger': 70}
# THRESHOLDS = {'warn': suggest_warn, 'danger': suggest_danger}
print('Ambang yang akan diekspor:', THRESHOLDS)

## 8. Fitur paling berpengaruh

Berguna sebagai bahan penjelasan ke juri.

In [ ]:
importance = pd.Series(model.feature_importances_, index=FEATURES).sort_values()
ax = importance.plot.barh(figsize=(7, 0.45 * len(FEATURES) + 1), color='#2b6cb0')
ax.set_xlabel('Feature importance')
ax.set_title(f'Fitur paling berpengaruh ({CHOSEN})')
plt.tight_layout()
plt.show()

## 9. Uji dengan URL di luar dataset

Dataset sering tidak mewakili kondisi nyata (rancangan bagian 15). Contoh: di beberapa dataset, URL legit hampir semuanya hanya halaman depan (`https://www.situs.com`), sehingga model belajar bahwa URL panjang = phishing.

Cek dengan URL legit yang punya path panjang, dan tambahkan URL dari kampus/instansimu sendiri.

In [ ]:
SANITY_URLS = {
    # Legit dengan path panjang: jangan sampai dianggap phishing
    'https://www.its.ac.id/informatika/id/beranda/': 0,
    'https://github.com/scikit-learn/scikit-learn/blob/main/README.rst': 0,
    'https://id.wikipedia.org/wiki/Institut_Teknologi_Sepuluh_Nopember': 0,
    'https://docs.python.org/3/library/urllib.parse.html#url-parsing': 0,
    'https://www.tokopedia.com/search?st=product&q=laptop%20gaming': 0,
    'https://news.ycombinator.com/item?id=1': 0,
    # Pola mencurigakan (contoh buatan, bukan situs nyata)
    'http://192.168.10.5/login/verify.php': 1,
    'http://secure-login-bank-update.example-verify.com/account/confirm': 1,
    'https://login.secure.account.verify.example.xyz/signin?session=8f7a6c': 1,
    'http://bank.example@203.0.113.7/login': 1,
    'http://login-kampus-verifikasi.free-host.test:8000/fake-login/': 1,
}

sanity = EX.features_frame(SANITY_URLS, FEATURES)
sanity_proba = positive_proba(model, sanity)
pd.DataFrame({
    'url': list(SANITY_URLS),
    'harapan': ['phishing' if v else 'legit' for v in SANITY_URLS.values()],
    'skor model (%)': np.floor(sanity_proba * 100 + 0.5).astype(int),
})

## 10. Simpan & ekspor ke ekstensi

Sel di bawah menyimpan `artifacts/model.joblib`, lalu menulis:
- `extension/model/model.json`: model yang dijalankan di browser
- `extension/model/test_vectors.json`: contoh URL + hasil Python, untuk dicocokkan di halaman uji ekstensi

Sebelum menulis, hasil JSON dicek harus sama persis dengan scikit-learn.

In [ ]:
metrics = {k: round(float(v), 4) for k, v in evaluate(model, X_test, y_test).items()}
bundle = {
    'model': model,
    'features': FEATURES,
    'thresholds': THRESHOLDS,
    'metrics': metrics,
    'dataset': {
        'file': DATA_PATH.name,
        'n_train': int(len(X_train)),
        'n_test': int(len(X_test)),
        'phishing_ratio': round(float(y.mean()), 4),
    },
    'created_at': datetime.datetime.now().isoformat(timespec='seconds'),
}
joblib.dump(bundle, ARTIFACTS / 'model.joblib')
print('Tersimpan:', ARTIFACTS / 'model.joblib')
metrics

In [ ]:
verify_X = X_test.sample(n=min(len(X_test), 2000), random_state=RANDOM_STATE)
payload = EX.export_bundle(bundle, verify_X=verify_X)

legit_urls = url_test[y_test == 0]
phish_urls = url_test[y_test == 1]
vector_urls = (
    legit_urls.sample(n=min(100, len(legit_urls)), random_state=RANDOM_STATE).tolist()
    + phish_urls.sample(n=min(100, len(phish_urls)), random_state=RANDOM_STATE).tolist()
    + list(SANITY_URLS)
)
EX.write_test_vectors(bundle, payload, vector_urls)

## 11. Langkah berikutnya

1. Buka `chrome://extensions` lalu klik ⟳ (reload) pada PhishShield.
2. Klik ikon PhishShield → **Pengaturan & model** → **Uji kecocokan model**. Semua baris harus cocok.
3. Coba skenario di `demo/README.md`, lalu buka situs-situs normal untuk mengecek false positive.
4. Kalau false positive terlalu banyak: naikkan ambang, tambah whitelist, atau latih ulang dengan data legit yang lebih beragam.